<a href="https://colab.research.google.com/github/nisaa783/scikit-learn-cookbook_Anisa-Khasanah/blob/main/Chapter_11_Novelty_and_Outlier_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Pendahuluan dan Perbedaan Outlier vs. Novelty Detection
Dalam analisis data dan machine learning, mendeteksi titik data yang menyimpang jauh dari pola umum sangat penting untuk menjaga kualitas model, mendeteksi kecurangan (fraud), mengidentifikasi kerusakan sensor, atau menemukan anomali pada log sistem. Secara konseptual, deteksi penyimpangan dibagi menjadi dua pendekatan:

Outlier Detection: Data pelatihan sudah mengandung data normal sekaligus pencilan (outliers). Tujuannya adalah menemukan titik data yang berbeda jauh dari mayoritas distribusi data tersebut.

# Novelty Detection: Data pelatihan diasumsikan bersih dan hanya berisi pola normal. Tujuannya adalah mendeteksi apakah data baru yang masuk saat fase pengujian merupakan pola baru yang belum pernah terlihat sebelumnya (novelty).

## 2. Algoritma Deteksi Anomali: Isolation Forest dan Local Outlier Factor
Dua pendekatan populer dalam scikit-learn untuk mendeteksi anomali adalah berbasis pohon partisi acak dan berbasis kepadatan lokal:

Isolation Forest: Memanfaatkan fakta bahwa data anomali jumlahnya sedikit dan memiliki nilai fitur yang sangat berbeda sehingga lebih mudah diisolasi. Algoritma ini membangun banyak isolation tree dengan memilih fitur dan nilai pemisah secara acak; titik data yang terisolasi dalam jumlah langkah pemisahan (path length) yang lebih sedikit cenderung merupakan anomali.

Local Outlier Factor (LOF): Mengukur penyimpangan kepadatan lokal suatu titik data dibandingkan dengan tetangga-tetangganya. Titik yang berada di area jauh lebih jarang (kepadatan jauh lebih rendah daripada tetangganya) akan memiliki skor LOF yang tinggi dan teridentifikasi sebagai pencilan.

Konsep & Rumus Matematika:Skor Anomali Isolation Forest berdasarkan panjang jalur rata-rata $E(h(x))$ pada pohon dengan ukuran sub-sampel $n$:$$s(x, n) = 2^{-\frac{E(h(x))}{c(n)}}$$di mana $c(n)$ adalah faktor normalisasi panjang jalur pencarian pada pohon biner:$$c(n) = 2H(n - 1) - \frac{2(n - 1)}{n}, \quad H(i) \approx \ln(i) + 0.5772156649$$Nilai $s(x, n)$ mendekati $1$ mengindikasikan anomali kuat, mendekati $0.5$ mengindikasikan data normal, dan jauh di bawah $0.5$ mengindikasikan data normal yang padat.Local Outlier Factor (LOF) membandingkan kepadatan jangkauan lokal (local reachability density, $\text{lrd}$) titik $A$ dengan rata-rata $\text{lrd}$ dari $k$ tetangga terdekatnya ($N_k(A)$):$$\text{LOF}_k(A) = \frac{\sum_{B \in N_k(A)} \frac{\text{lrd}_k(B)}{\text{lrd}_k(A)}}{\vert{}N_k(A)\vert{}}$$Nilai $\text{LOF}_k(A) \approx 1$ menunjukkan kepadatan sebanding dengan tetangga (normal), sedangkan nilai jauh lebih besar dari $1$ menandakan outlier.

## 3. Contoh Program Penerapan 1: Deteksi Pencilan Menggunakan Isolation Forest
Contoh program pertama ini membuat data normal ditambah beberapa titik pencilan ekstrem, lalu melatih IsolationForest untuk mengidentifikasi data normal (1) dan pencilan (-1).



In [1]:
import numpy as np
from sklearn.ensemble import IsolationForest

# 1. Membuat data normal (2D) dan menambahkan beberapa titik pencilan ekstrem
rng = np.random.RandomState(42)
X_normal = 0.4 * rng.randn(120, 2)
X_outliers = np.array([
    [4.5, 4.5],
    [-4.2, 4.0],
    [4.0, -4.5],
    [-3.8, -4.0],
    [5.0, -0.5]
])
X = np.vstack([X_normal, X_outliers])

# 2. Melatih Isolation Forest (contamination menandakan proporsi perkiraan outlier)
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=len(X_outliers) / len(X),
    random_state=42
)
preds = iso_forest.fit_predict(X)
scores = iso_forest.decision_function(X)

# Pada scikit-learn: -1 berarti outlier/anomali, 1 berarti data normal
outlier_indices = np.where(preds == -1)[0]
print(f"Total sampel: {len(X)}")
print(f"Jumlah terdeteksi sebagai outlier (-1): {len(outlier_indices)}")
print(f"Indeks outlier yang terdeteksi: {outlier_indices.tolist()}")

print("\n5 titik data dengan skor anomali terendah (paling mencurigakan):")
lowest_idx = np.argsort(scores)[:5]
for idx in lowest_idx:
    print(f"Indeks {idx:3d} | Koordinat: {X[idx].round(3)} | Label: {preds[idx]:2d} | Score: {scores[idx]:.4f}")

Total sampel: 125
Jumlah terdeteksi sebagai outlier (-1): 5
Indeks outlier yang terdeteksi: [120, 121, 122, 123, 124]

5 titik data dengan skor anomali terendah (paling mencurigakan):
Indeks 123 | Koordinat: [-3.8 -4. ] | Label: -1 | Score: -0.2257
Indeks 121 | Koordinat: [-4.2  4. ] | Label: -1 | Score: -0.2194
Indeks 122 | Koordinat: [ 4.  -4.5] | Label: -1 | Score: -0.2125
Indeks 120 | Koordinat: [4.5 4.5] | Label: -1 | Score: -0.2107
Indeks 124 | Koordinat: [ 5.  -0.5] | Label: -1 | Score: -0.1380


## 4. Contoh Program Penerapan 2: Deteksi Anomali Berbasis Kepadatan dengan Local Outlier Factor

Contoh program kedua ini menggunakan LocalOutlierFactor pada dataset load_breast_cancer yang telah diskalakan untuk mendeteksi observasi dengan kepadatan lokal yang menyimpang dari tetangganya.

In [2]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

# 1. Memuat dataset dan menstandarkan skala fitur
data = load_breast_cancer()
X_scaled = StandardScaler().fit_transform(data.data)

# 2. Inisialisasi dan menjalankan Local Outlier Factor (outlier detection mode)
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
y_pred = lof.fit_predict(X_scaled)

# negative_outlier_factor_ bernilai negatif; semakin kecil (makin negatif), makin kuat indikasi outlier
lof_scores = lof.negative_outlier_factor_

n_outliers = np.sum(y_pred == -1)
print(f"Jumlah sampel terdeteksi outlier: {n_outliers} dari {len(X_scaled)} sampel")

# Menampilkan 5 sampel dengan negative_outlier_factor_ paling rendah
worst_idx = np.argsort(lof_scores)[:5]
print("\n5 Sampel dengan LOF Score Paling Ekstrem:")
for idx in worst_idx:
    print(f"Sampel #{idx:3d} | Prediksi: {y_pred[idx]:2d} | LOF Factor: {lof_scores[idx]:.4f}")

Jumlah sampel terdeteksi outlier: 29 dari 569 sampel

5 Sampel dengan LOF Score Paling Ekstrem:
Sampel #212 | Prediksi: -1 | LOF Factor: -2.8929
Sampel #461 | Prediksi: -1 | LOF Factor: -2.5342
Sampel #152 | Prediksi: -1 | LOF Factor: -2.3867
Sampel #213 | Prediksi: -1 | LOF Factor: -2.3709
Sampel #192 | Prediksi: -1 | LOF Factor: -2.0688
